# 06 · Error analysis

**What this notebook decides.** What to build next. An aggregate accuracy tells
you whether you improved; only the failure structure tells you what to do about
it.

The first split is the one that matters most:

- **Invalid SQL** — the model produced something that will not run. A parsing,
  formatting or extraction problem.
- **Valid but wrong** — the model produced runnable SQL answering a different
  question. A reasoning or schema-linking problem.

These have almost nothing in common, and lumping them into one error rate hides
which one you actually have.

In [ ]:
import pandas as pd

from qwen_text2sql.reporting import (
    artifact,
    environment_report,
    error_breakdown,
    load_records,
    preflight,
    project_root,
)
from qwen_text2sql.reporting.style import use_house_style

ROOT = project_root()
PALETTE = use_house_style("light")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
ROOT

In [ ]:
report = environment_report()
print(f"commit   {report['git_commit']}")
print(f"python   {report['python']}")
print(f"platform {report['platform']}")
pd.Series(report["packages"], name="version").to_frame()

In [ ]:
pd.DataFrame(preflight("lora_records"))

In [ ]:
records = load_records(artifact("lora_records").require())
print(f"{len(records):,} evaluated examples across {records['db_id'].nunique()} databases")

## 1 · The first split

In [ ]:
invalid = ~records["valid_sql"]
valid_but_wrong = records["valid_sql"] & ~records["execution_match"]
correct = records["execution_match"]

split = pd.DataFrame(
    {
        "outcome": ["correct", "valid but wrong", "invalid SQL"],
        "count": [int(correct.sum()), int(valid_but_wrong.sum()), int(invalid.sum())],
    }
).assign(share=lambda frame: frame["count"] / len(records))
split.style.format({"share": "{:.1%}"})

**Which number is larger changes the next experiment.**

If *invalid SQL* dominates, the win is in decoding and extraction — constrained
decoding against the schema, or a stricter SQL extractor. Model capacity is not
the bottleneck.

If *valid but wrong* dominates, the model can write SQL but is picking the wrong
tables, columns or aggregation. That is a schema-linking and reasoning problem,
and more adapter rank is unlikely to fix it on its own.

## 2 · Failure categories

In [ ]:
from qwen_text2sql.reporting import figures

breakdown = error_breakdown(records)
display(breakdown.style.format({"share": "{:.1%}"}))
if bool((breakdown["error_kind"] != "none").any()):
    display(figures.error_breakdown(breakdown))

`missing_table` and `missing_column` are the schema-linking signal: the model
referenced something that does not exist. A high rate here points at schema
serialisation — the model may not be seeing the columns it needs, which
[`01_data_audit`](01_data_audit.ipynb) checks against the context budget.

## 3 · Is failure concentrated or spread?

In [ ]:
per_db = (
    records.assign(failed=~records["execution_match"])
    .groupby("db_id")
    .agg(examples=("example_id", "size"), failures=("failed", "sum"))
    .assign(failure_rate=lambda frame: frame["failures"] / frame["examples"])
    .sort_values("failure_rate", ascending=False)
)

total_failures = int((~records["execution_match"]).sum())
concentration = pd.Series(
    {
        "databases": len(per_db),
        "databases with zero failures": int((per_db["failures"] == 0).sum()),
        "share of failures in worst 5 databases": (
            per_db["failures"].head(5).sum() / total_failures if total_failures else 0.0
        ),
    },
    name="value",
).to_frame()
display(concentration)
display(per_db.head(10).style.format({"failure_rate": "{:.1%}"}))

In [ ]:
crosstab = pd.crosstab(records["db_id"], records["error_kind"])
crosstab = crosstab.loc[crosstab.sum(axis=1).sort_values(ascending=False).index]
crosstab.head(15)

A database where nearly every failure is `missing_column` is a different problem
from one where they are spread across categories. The first suggests its schema
is being truncated or serialised badly; the second suggests the questions are
genuinely harder.

## 4 · Reading actual failures

Aggregates stop being useful at some point and you have to read queries. These
are valid-but-wrong cases — the interesting ones.

In [ ]:
def show_failures(frame: pd.DataFrame, limit: int = 5) -> None:
    """Print gold and predicted SQL side by side for manual inspection."""
    rule = "-" * 78
    for row in frame.head(limit).itertuples(index=False):
        print(rule)
        print(f"db={row.db_id}  example={row.example_id}  error={row.error_kind}")
        print(rule)
        print("GOLD      :", " ".join(str(row.gold_sql).split()))
        print("PREDICTED :", " ".join(str(row.predicted_sql).split()))
        print()


show_failures(records[valid_but_wrong])

### Recording suspected annotation errors

BIRD contains reference queries that do not answer their question. Rule 7 says
those are counted separately from model errors, never folded into them. When
manual inspection finds one, record its `example_id` here so the exclusion is
explicit and reviewable rather than an undocumented filter.

In [ ]:
# Populate after manual review. Keep the reason: an unexplained exclusion is
# indistinguishable from cherry-picking.
SUSPECTED_ANNOTATION_ERRORS: dict[str, str] = {
    # "abc123": "gold query returns all customers; question asks only for active ones",
}

if SUSPECTED_ANNOTATION_ERRORS:
    flagged = records[records["example_id"].isin(SUSPECTED_ANNOTATION_ERRORS)]
    adjusted = records[~records["example_id"].isin(SUSPECTED_ANNOTATION_ERRORS)]
    print(f"Excluding {len(flagged)} suspected annotation error(s).")
    print(f"Execution accuracy as reported : {records['execution_match'].mean():.1%}")
    print(f"Execution accuracy excluding   : {adjusted['execution_match'].mean():.1%}")
    print("\nReport BOTH numbers. The adjusted figure is not the headline result.")
else:
    print("No suspected annotation errors recorded yet.")

## Reading this analysis

**What it shows.** Where failures concentrate — by validity, by category, by
database — and enough worked examples to form a hypothesis.

**What it does not show.** Causation. A high `missing_column` rate is consistent
with truncated schemas, with poor schema linking, and with genuinely ambiguous
questions. Distinguishing them takes a targeted experiment, which is what
[`00_research_protocol`](00_research_protocol.ipynb)'s proposal process is for.

---

### Provenance

Every number above is produced by code in `src/qwen_text2sql/`, which is unit
tested and type checked. This notebook orchestrates and interprets; it does not
define the analysis. If a figure here looks wrong, the fix belongs in `src/` and
comes with a test.